<a href="https://colab.research.google.com/github/hmmnyamminji/bootcampproject/blob/main/Chapter08_%ED%85%8D%EC%8A%A4%ED%8A%B8%EB%B6%84%EC%84%9D_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
review_df = pd.read_csv('/content/drive/MyDrive/KWU/ML/Data/labeledTrainData.tsv',
                        header=0, sep='\t', quoting=3)# 따옴표를 특별 처리 없이 그대로 읽음
print('데이터 shape:', review_df.shape)
print('레이블 분포')
print(review_df['sentiment'].value_counts())
review_df.head(3)

데이터 shape: (25000, 3)
레이블 분포
sentiment
1    12500
0    12500
Name: count, dtype: int64


,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."


In [ ]:
import re

In [ ]:
# 텍스트 클렌징(Text Cleansing)
# HTML 형식으로 저장 -> 태그 제거
review_df['review'] = review_df['review'].str.replace('<br />', ' ', regex=False) # 정규식이 아닌 문자 그대로 치환

# 숫자, 특수문자는 감성 분석에 노이즈 -> 제거
review_df['review'] = review_df['review'].apply(lambda x: re.sub('[^a-zA-Z]', ' ', x))
# 입력으로 받은 x 문자열에 대해


In [ ]:
from sklearn.model_selection import train_test_split

#학습(70%) / 테스트(30%) 분할
class_df = review_df['sentiment'] # 레이블: 0 부정 또는 1 긍정
feature_df = review_df.drop(['id', 'sentiment'], axis=1) # 피처: 리뷰

X_train, X_test, y_train, y_test = train_test_split(
    feature_df, class_df, test_size=0.3, random_state=156
)
print('학습셋:', X_train.shape, '테스트셋:', X_test.shape)

학습셋: (17500, 1) 테스트셋: (7500, 1)


In [ ]:
# 지도 학습 감성 분석: Count vs TF-IDF + 로지스틱 회귀
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score

# Count + 로지스틱 회귀
pipeline_cnt = Pipeline([
    ('cnt_vect', CountVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])
pipeline_cnt.fit(X_train['review'], y_train)
pred_cnt = pipeline_cnt.predict(X_test['review'])
prob_cnt = pipeline_cnt.predict_proba(X_test['review'])[:,1] #긍정(클래스 1)확률 -> AUC 계산에 사용
print(f'Count 기반 정확도: {accuracy_score(y_test, pred_cnt):.3f}, AUC: {roc_auc_score(y_test, prob_cnt):.3f}')
# TF-IDF + 로지스틱 회귀
pipeline_tfidf = Pipeline([
    ('tfidf_vect', CountVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])
pipeline_tfidf.fit(X_train['review'], y_train)
pred_tfidf = pipeline_tfidf.predict(X_test['review'])
prob_tfidf = pipeline_tfidf.predict_proba(X_test['review'])[:,1] #긍정(클래스 1)확률 -> AUC 계산에 사용
print(f'Count 기반 정확도: {accuracy_score(y_test, pred_tfidf):.3f}, AUC: {roc_auc_score(y_test, prob_tfidf):.3f}')

Count 기반 정확도: 0.886, AUC: 0.950
Count 기반 정확도: 0.886, AUC: 0.950


In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('sentiwordnet')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package sentiwordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/sentiwordnet.zip.


True

In [ ]:
from nltk.corpus import wordnet as wn
from nltk.corpus import sentiwordnet as swn
from nltk.stem import WordNetLemmatizer
from nltk import sent_tokenize, word_tokenize, pos_tag # 텍스트 내의 각 단어에 품사 태그를 부여

In [ ]:
def penn_to_wn(tag): #NLTK pos_tag()는 penn Treebank 태그 반환 (JJ=형용사, NN=명사, VB=동사 등)
  if tag.startswith('J'): return wn.ADJ #WordNet 품사 태그 변환
  elif tag.startswith('N'): return wn.NOUN
  elif tag.startswith('R'): return wn.ADV
  elif tag.startswith('V'): return wn.VERB
  return None #해당 없는 품사: 감성 분석 제외


def swn_polarity(text):
  sentiment = 0.0 #감성 점수 합산 변수
  tokens_count = 0 #분석한 단어 수
  lemmatizer = WordNetLemmatizer()

  for raw_sentence in sent_tokenize(text): #문장 분리
    for word, tag in pos_tag(word_tokenize(raw_sentence)): #pos_tag: [(단어, 품시태그)...]형태로 반환
      wn_tag = penn_to_wn(tag)

      if wn_tag not in(wn.NOUN, wn.ADJ, wn.ADV): #명사/형용사/부사만 감성 분석 대상, 동사는 행동 표현이라서 제외
        continue

      lemma = lemmatizer.lemmatize(word,pos=wn_tag) #원형 추출: playing -> play, better(adj) -> good

      synsets = wn.synsets(lemma, pos=wn_tag) #단어의 동의 집합(Synonym Set)
      if not synsets: #동의어 집합이 없다면
        continue

      swn_synset = swn.senti_synset(synsets[0].name()) #가장 일반적인 의미의 synset
      #첫 번째 synset의 감성 점수 조회
      sentiment += swn_synset.pos_score() - swn_synset.neg_score()
      tokens_count += 1

  if not tokens_count:
    return 0 #분석 가능한 단어 없으면 중립(부정) 반환
  return 1 if sentiment >= 0 else 0 #클래스 1은 긍정, 클래스 0은 부정
print('SentiWordNet 감성 분성')
review_df['swn_preds'] = review_df['review'].apply(swn_polarity)

print(confusion_matrix(review_df['sentiment'], review_df['swn_preds']))
print(f'정확도: {accuracy_score(review_df['sentiment'], review_df['swn_preds']):.4f}')
print(f'정밀도: {precision_score(review_df['sentiment'], review_df['swn_preds']):.4f}')
print(f'재현율: {recall_score(review_df['sentiment'], review_df['swn_preds']):.4f}')

SentiWordNet 감성 분성
[[7669 4831]
 [3635 8865]]
정확도: 0.6614
정밀도: 0.6473
재현율: 0.7092


In [ ]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

senti_analyzer = SentimentIntensityAnalyzer() #VADER 구현하는 객체 생성

sample = senti_analyzer.polarity_scores(review_df['review'][0])
print('VADER 점수 예시:',sample)
#neg: 부정 비율/neu:중립 비율/pos: 긍정 비율
#compound: -1(완전 부정) ~ +1(완전 긍정) 종합 점수 (핵심 지표)

VADER 점수 예시: {'neg': 0.13, 'neu': 0.743, 'pos': 0.127, 'compound': -0.7943}


In [ ]:
def vader_polarity(review, threshold=0.1):
  scores = senti_analyzer.polarity_scores(review)
  agg_score = scores['compound']
  return 1 if agg_score >= threshold else 0

review_df['vader_preds'] = review_df['review'].apply(
    lambda x: vader_polarity(x, 0.1)
)

y_target = review_df['sentiment'].values
vader_preds = review_df['vader_preds'].values

print('VADER 감성 분석')
print(confusion_matrix(y_target, vader_preds))
print(f'정확도: {accuracy_score(y_target, vader_preds):.4f}')
print(f'정밀도: {precision_score(y_target, vader_preds):.4f}')
print(f'재현율: {recall_score(y_target, vader_preds):.4f}')

#theshold 임계값을 낮추면 재현율 상승, 정밀도 하락

VADER 감성 분석
[[ 6747  5753]
 [ 1858 10642]]
정확도: 0.6956
정밀도: 0.6491
재현율: 0.8514
